# PDF Chunker
This notebook walks through splitting budget PDFs into chunks using budget code headers
(e.g. `QB0-9QGDTAC-OW`) as delimiters. The output is a parquet file with one row per chunk.

Make sure you have added your PDFs to the `pdfs/` folder before running.

Note: Feel free to change the parsers used to convert the pdf into a string. I only used `markitdown` because it is lightweight and I didnt know what types of machines that this will be run on.

In [1]:
from pathlib import Path

import pandas as pd
from markitdown import MarkItDown
from tqdm import tqdm

from src.chunking import chunk_text_by_header_codes

## Configuration
The pattern matches budget codes in the format `QB0-9QGDTAC-OW`:
- 3 alphanumeric characters, a hyphen, 7 alphanumeric characters, a hyphen, 2 alphanumeric characters

In [5]:
PATTERN = r"\b[A-Za-z0-9]{3}-[A-Za-z0-9]{7}-[A-Za-z0-9]{2}\b"
OUTPUT_PATH = Path("results")
DF_FILENAME = "chunked_text.parquet"

OUTPUT_PATH.mkdir(exist_ok=True)

# Load PDFs
Collect all PDFs from the `pdfs/` folder and convert them to plain text using markitdown.

In [6]:
script_path = Path(".")
pdf_folder_path = script_path / "pdfs"

if not pdf_folder_path.exists():
    pdf_folder_path.mkdir(exist_ok=True)
    raise FileNotFoundError(f"Please add PDFs to the {pdf_folder_path} folder and run again.")

pdfs = [file for file in pdf_folder_path.glob("*.pdf") if file.is_file()]

if not pdfs:
    raise FileNotFoundError(f"No PDFs found in {pdf_folder_path}. Please add PDFs and run again.")

print(f"Found {len(pdfs)} PDF(s):")
for pdf in pdfs:
    print(f"  - {pdf.name}")

Found 2 PDF(s):
  - VR 2025 1605 MED.0176-1 BBT BU 2024 Onroerend erfgoed - mededeling.pdf
  - VR 2025 1605 MED.0166-1 BBT BU 2024 Energie en Klimaat - mededeling.pdf


In [7]:
# Start converting PDFs to text
md_converter = MarkItDown(enable_plugins=False)
results = [
    md_converter.convert(file).text_content
    for file in tqdm(pdfs, desc="Converting PDFs to text")
]

# Peek at the first 500 characters of the first converted PDF
print(results[0][:500])

Converting PDFs to text: 100%|██████████| 2/2 [00:08<00:00,  4.07s/it]

Beleids- en begrotingstoelichting

ONROEREND ERFGOED

Begrotingsuitvoering 2024

Ben Weyts

Vlaams Minister van Begroting en Financiën, Vlaamse Rand, Onroerend Erfgoed en
Dierenwelzijn

1

VOORBLAD

I. INHOUDSTAFEL

II. INLEIDING DOOR DE MINISTER ............................................................ 4

III. SAMENVATTING ................................................................................ 4

IV. TRANSVERSALE, HORIZONTALE EN OVERKOEPELENDE STRATEGISCHE
DOELSTELLINGEN ..........


## Chunk the text
Split each PDF's text into chunks using budget code headers as delimiters.
Each chunk runs from one budget code header to the next.

In [8]:
df = pd.concat(
    [
        chunk_text_by_header_codes(text, PATTERN, file.name)
        for text, file in zip(tqdm(results, desc="Chunking text"), pdfs)
    ]
)

print(f"Found {len(df)} chunks across {df['filename'].nunique()} file(s)")
df.head()


Chunking text: 100%|██████████| 2/2 [00:00<00:00, 251.25it/s]

Found 50 chunks across 2 file(s)


,code,chunk,filename
0,QB0-9QGDTAC-OW,QB0-9QGDTAC-OW – DE ZORG VOOR ONROEREND ERFGOE...,VR 2025 1605 MED.0176-1 BBT BU 2024 Onroerend ...
1,QG0-9QGDAAA-OW,QG0-9QGDAAA-OW – DE ZORG VOOR ONROEREND ERFGOE...,VR 2025 1605 MED.0176-1 BBT BU 2024 Onroerend ...
2,QG0-9QGDTAB-OI,QG0-9QGDTAB-OI – DE ZORG VOOR ONROEREND ...,VR 2025 1605 MED.0176-1 BBT BU 2024 Onroerend ...
3,QB0-1QGD4AC-WT,QB0-1QGD4AC-WT – DE ZORG VOOR ONROEREND ERFGOE...,VR 2025 1605 MED.0176-1 BBT BU 2024 Onroerend ...
4,QG0-1QGD2AA-WT,QG0-1QGD2AA-WT – DE ZORG VOOR ONROEREND ERFGOE...,VR 2025 1605 MED.0176-1 BBT BU 2024 Onroerend ...


In [ ]:
# Print first chink
print(df["chunk"].iloc[0])

QB0-9QGDTAC-OW – DE ZORG VOOR ONROEREND ERFGOED VOOR IEDEREEN
VANZELFSPREKEND MAKEN (FONDS HANDHAVING ONROEREND ERFGOED)

Korte inhoud begrotingsartikel:
Ontvangsten op grond van het handhavingsluik van het Onroerenderfgoeddecreet
van 12 juli 2013 en zijn uitvoeringsbesluiten en de handhavingsbepalingen in de
decreten,  vermeld  in  artikel  12.2.1  van  het  Onroerenderfgoeddecreet  worden
aangerekend in het begrotingsfonds ‘Fonds handhaving onroerend erfgoed’ binnen
de begrotingsstructuur van het departement Omgeving. Het ontvangstenartikel is
QB0-9QGDTAC-OW. Dit wordt gespiegeld door het uitgavenartikel QB0-1QGD4AC-
WT.  Ik  ben  als  minister  bevoegd  voor  onroerend  erfgoed  verantwoordelijk  voor
deze materie.

Begrotingsuitvoering:

BA 2024

BA-JR 2024

BU 2024

(duizend euro)

AO

TO

LO

0

0

0

250

250

589

0

0

0

Inhoudelijke toelichting:
De gerealiseerde toegewezen ontvangst bestaat uit:

-  436 keuro schadevergoedingen;
-  138 keuro dwangsommen;
-  11 keuro vergoedi

## Save results
Save the chunks to a parquet file in the `results/` folder.

In [ ]:
df.to_parquet(OUTPUT_PATH / DF_FILENAME, index=False)
print(f"Saved to {OUTPUT_PATH / DF_FILENAME}")